# Рекуррентные нейронные сети для NLP

К концу 2000-х языковое моделирование в основном опиралось на n-граммы, а нейросетевая языковая модель с фиксированным окном [(Bengio et al., 2003)](https://jmlr.org/papers/v3/bengio03a.html) уже показала силу распределённых представлений слов (прародителя эмбедингов). Главная проблема - обе конструкции ограничены, контекст обрезан заранее заданным окном. 

**Рекуррентные нейронные сети** (recurrent neural networks, RNN) [(Elman, 1990)](https://doi.org/10.1207/s15516709cog1402_1) сняли это ограничение: состояние фиксированного размера, обновляемое на каждом шаге, в принципе способно нести информацию о префиксе любой длины. Рекуррентная языковая модель [(Mikolov et al., 2010)](https://doi.org/10.21437/Interspeech.2010-343) первой уверенно обошла n-граммы, и примерно на десятилетие (2010–2017) RNN стали доминирующей архитектурой NLP. 

В этой главе рассмотрим историю развития данного типа сетей: от простых рекуррентных single cell моделей к (LSTM, GRU), encoder–decoder архитекутуре и механизхму Attnetion (из которого вырастут совеременные Трансформеры), к попытке вернуть рекуррентность в лице state-space моделей

## Введение

### Отказ от гипотезы i.i.d.

Классическое обучение с учителем предполагает выборку независимых одинаково распределённых пар $(x_i, y_i)$ фиксированной размерности. Текст нарушает оба допущения: токены внутри последовательности зависимы (порядок несёт смысл), а длина переменная. Вероятностная постановка задаётся цепным правилом

$$p(x_1,\dots,x_T)=\prod_{t=1}^{T}p(x_t\mid x_1,\dots,x_{t-1}),$$

то есть моделировать нужно условные распределения с неограниченно растущим контекстом. N-граммовые модели делают это в условиях марковского допущения (см. I главу) порядка $n-1$, . Нейросетевая язвковая модель Бенджио обрезает контекст фиксированным окном. Нужен механизм, сворачивающий префикс произвольной длины в представление постоянного размера

### Типы задач по форме входа и выхода

По соотношению входа и выхода выделяют четыре режима: 
- many-to-one — последовательность в один выход (классификация тональности, детекция спама);
- one-to-many — генерация последовательности из одного входа (порождение текста по затравке, описание изображения);
- синхронный many-to-many — выход на каждом шаге, $T_{вх}=T_{вых}$ (частеречная разметка, NER);
- асинхронный many-to-many — длины входа и выхода различаются (перевод, суммаризация), что потребует отдельной архитектуры encoder–decoder.

Все четыре режима обслуживаются одной и той же рекуррентной ячейкой — меняется только то, где снимается выход.

<img src="img/rnn_output_types.png" width=600>

### Полносвязные сети против рекуррентных

Полносвязная сеть требует вход фиксированной размерности: текст приходится обрезать или скользить окном, число параметров растёт с шириной окна, а паттерн, выученный в позиции 5, не переносится в позицию 50. RNN применяет одни и те же матрицы на каждом шаге — разделение весов во времени, прямой аналог разделения весов свёртки по пространству. Следствия: число параметров не зависит от $T$; инвариантность к позиции паттерна; последовательность любой длины обрабатывается за один проход.

### Скрытое состояние как память

Информация о префиксе аккумулируется в векторе $h_t\in\mathbb{R}^{d}$ — скрытом состоянии. Модель делает марковское допущение на уровне состояния: $p(x_{t+1}\mid x_{\le t})\approx p(x_{t+1}\mid h_t)$, то есть $h_t$ — обучаемая достаточная статистика префикса. Это сжатие с потерями: сколь угодно длинная история упаковывается в $d$ чисел. Отсюда и сила RNN (константная память на инференсе), и её главный будущий конфликт — информационное бутылочное горлышко, которое проявится в seq2seq.

Условное ведро, в которое складываете поленую информацию. В какой-то момент ведро переполнится

## Recurrent Neural Networks

С начала 1980-ых начало активно развиваться направление рекуррентных сетей. В 1982 появилась сеть Хопфилда - полносвязная однослойная сеть, способная запоминать паттерны. Ассоциативной памяти. Обучалась рекуррентно. Чуть позже появилась построенная на том же принципе [машина Больцмана](https://en.wikipedia.org/wiki/Boltzmann_machine)

В 1985 году был описан метод обучения нейросетей на основе обратного распространения ошибки ([error backpropagation](https://en.wikipedia.org/wiki/Backpropagation)) - он обобщил обучение на многослойные сети произвольной глубины. До этого сети обучались более простыми подходами, например, Hebbian Learning, Delta Rule. Идея разрабатывалась параллельно несколькими учеными, включая Яна Лекуна, наиболее цитируемая работа принадлежит Румельхарту.

В 1986 году Джордан (Michael I. Jordan) предложил рекурентную модель для обработки последовательностей. В рамках модели выход сети (Output) подавался на вход следующему слою (часть входа, обозначенная как Context). Далее модель комбинировала его с обычным входом (часть входа Input). Фактически тут одним из первых был реализован мезанизм запоминания

<img src="img/jordan_rnn.png" width=250>

В 1988 году Джефри Элман (Jeffrey Elman), вдохновился идеей Джордана, но предложил вместо выходов предыдущего слоя подавать на вход его скрытые состояния. Идея в том, что скрытые состояния хранят существенно больше внутренней информации, чем выходной слой. Так появилась сеть Элмана:

$$\begin{cases}h_t=f\big(W_{xh}\,x_t+W_{hh}\,h_{t-1}+b_h\big),\qquad \\ y_t=g\big(W_{hy}\,h_t+b_y\big),\end{cases}$$

где $x_t\in\mathbb{R}^{e}$ — вход (например, эмбеддинг токена)<Br>$W_{xh}\in\mathbb{R}^{d\times e}$, $W_{hh}\in\mathbb{R}^{d\times d}$ - обучаемые матрицы преобразования<br>$f$ — функция активации для состояния (как правило tanh)<br>$g$ - функция активации для выхода: softmax для языковой модели, сигмоида для бинарной классификации

Эта архитектура стала стандартом для рекурентных сетей того времени

<img src="img/elman_rnn.jpeg" width=300>

Любое рекуррентное выражение вида $x = f(x)$ можно полностью развернуть до своего начала $f(0)$, выписав его одним выражением, для рекуррентной ети это называют __Unfolding__. Так, если обрабатывается последовательность из T токенов, в развернутом варианте это будет единый граф вычислений глубины $T$ с большим кол-вом повторяющихся весов

<img src="img/unfold.png" width=300>

Видна хараткерная особенность. Многократное повторение умножения на одну и ту же матрицу обнажает одновременно<br> 
а) высокую выразительность внутренних представлений - у нас много нелинейных активаций<br>
б) большое кол-во проблем и сложностей обучения, главным образом, затухание и экспоненциальный взрыв градиентов

Выбор функции активации:
- гиперболический тангенс tanh() - стандартная функция активации для RNN. Выходной сигнал ограничен интервалом $(-1,1)$ => вектор состояния не может неограниченно расти, а также центрирован вокруг нуля. Мы таким образом обезопашиваем себя от взрыва градиента, но поскольку $|\tanh'|\le 1$ провоцируем затухание градиента<br><br>
- ReLU не насыщается и держит градиент 1 на активной части, но рекуррентное применение неограниченной функции взрывает активации при $\sigma_1(W_{hh})>1$<br><br>
- В более поздней работе [(Le et al., 2015)](https://arxiv.org/abs/1504.00941) была сделана попытка предложить компромиссное решение, модель **IRNN** : ReLU-сеть с инициализацией $W_{hh}=I$, $b=0$ на старте просто копирует состояние и на длинных зависимостях сопоставима с LSTM

Чем инициализировать начальное состояние
- $h_0=0$ — стандарт: до начала текста контекста нет
- обучаемый $h_0$ полезен на коротких последовательностях, где старт вносит заметный вклад: выучивается априорный контекст
- cлучайный шум в $h_0$ при обучении — лёгкая регуляризация, снижающая зависимость от начала
- $h_0$ очередного чанка равен $h_T$ предыдущего (stateful-режим) — используется при обучении на длинных потоках усечённым BPTT (см. ниже)

## Backpropagation Throught Time

Самым сложным было понять, как такие рекуррентное сети правильно обучать

**BPTT** (backpropagation through time) [(Werbos, 1990)](https://doi.org/10.1109/5.58337) — обычный backpropagation, применённый к развёрнутому графу

Рассмотрим развернутый граф $$
y = f\Big( W \cdot f\big( W \cdot f(W \cdot h_0) \big) \Big)
$$

По формуле полной проивзодной суммируются частные производные


Для суммарных потерь $L=\sum_t L_t$ вклад шага $t$ в градиент по $W_{hh}$ собирается со всех предшествующих позиций $k\le t$:

$$\frac{\partial L_t}{\partial W_{hh}}=\sum_{k=1}^{t}\frac{\partial L_t}{\partial h_t}\left(\prod_{i=k+1}^{t}\frac{\partial h_i}{\partial h_{i-1}}\right)\frac{\partial^{+} h_k}{\partial W_{hh}},$$

где $\partial^{+}h_k/\partial W_{hh}$ — «немедленная» производная при замороженном $h_{k-1}$. Вся динамика обучения спрятана в произведении якобианов соседних шагов

В отличие от обычной глубокой, в рекуррентной сети матрица одна и та же на всех шагах ($W_{hh}$) => при подсчете градиента  суммировать

### Проблема устойчивости
Бенжио с коллегами исследовал возможности [(Bengio et al., 1994)](https://doi.org/10.1109/72.279181). Авторы показали, что RNN может надёжно выучивать зависимости длиной не более 5–20 шагов

Один шаг сети - это композиция двух преобразований: линейной части $z = W_{hh}h_{i-1} + W_{hx} x_i + b$ и функции активации $\sigma(z)$. Соотвественно, градиент этой композиции (как меняется выход $h_i$ при изменении входа $h_{i-1}$) - произведение двух производных: $W$ и $\sigma'(x)$.

Посольку функция активации, в отличие от линейного преобразования, поэлементная => выражается диагнональной матрицей

$$\frac{\partial h_i}{\partial h_{i-1}}=\operatorname{diag}\big(f'(a_i)\big)\,W_{hh}.$$

У нас произведение из $t - k$ таких шагов, поэтому $$\prod_{i=k+1}^{t}\frac{\partial h_i}{\partial h_{i-1}} = \prod_{i=k+1}^{t} \operatorname{diag}\big(f'(a_i)\big)\,W_{hh}$$ 

Напомним, что такое спектральная норма матрицы (иногда называют $L_2$ норму, не путать с Евклидовой):
$$
\|A\|_2 = \max_{x \neq 0} \frac{\|A x\|_2}{\|x\|_2}
$$

Максимум этой нормы достигается на макссимальном собственном значении $\sigma_{max}$

Градиент от потерь на шаге $t$ до состояния шага $k$ проходит через $t-k$ таких множителей и ведёт себя как степень матрицы: норма произведения ограничена

$$\Big\lVert\prod_{i=k+1}^{t}\frac{\partial h_i}{\partial h_{i-1}}\Big\rVert\le\big(\gamma\,\sigma_1(W_{hh})\big)^{\,t-k},$$

где $\sigma_1$ — наибольшее сингулярно число, а $\gamma=\sup|f'|$: 1 для tanh, 1/4 для сигмоиды

При $\sigma_1<1/\gamma$ вклад далёких шагов затухает экспоненциально. В линейном приближении судьбу решает спектральный радиус: $\rho(W_{hh})<1$ — затухание, $\rho>1$ — возможен взрыв

В работе [(Pascanu et al., 2013)](https://arxiv.org/abs/1211.5063) дали более глубокое описание фундаментальных ограничений базовой RNN модели Элмана. Они рассмотрели проблему сразу с трех позиций:
в дополнение к математике собственных значений с позиции динамической системы, а также геометрической

Там же они предложили два практических инструмента: "gradient clipping" для  борьбы с взрывающимся градиентом и регуляризация для борьбы с vanishing gradient

## Методы стабилизации обучения

__Gradient clipping__<br>
Gradient clipping = искусственное ограничение слишком большого градиента, возникающего в процессе обучения:
- по норме<br>если градиент слишком большой $\lVert g\rVert>\theta$, то он $g\leftarrow\theta\,g/\lVert g\rVert$ — нормируется, типичные $\theta\in[1,5]$<br><br>
- по отдельным параметрам<br>$g_i\leftarrow\operatorname{clip}(g_i,-\theta,\theta)$ — так проще, но искажает направление градиента

__Регуляризация__<br>
Добавляем в функцию потерь штраф за слишком большое или слишком маленькое изменение градиента

$$
\Omega = \sum_{t} \left( \frac{ \left\| \frac{\partial L}{\partial \mathbf{x}_{t+1}} \cdot \frac{\partial \mathbf{x}_{t+1}}{\partial \mathbf{x}_t} \right\| }{ \left\| \frac{\partial L}{\partial \mathbf{x}_{t+1}} \right\| } - 1 \right)^2
$$

Дробь отражает отношение градиента на шаге t к градиенту на шаге t+1. Просуммировано по всем расстояниям

**Truncated BPTT** [(Williams & Peng, 1990)](https://doi.org/10.1162/neco.1990.2.4.490)<br>длинный поток режется на чанки по $k$ шагов; состояние переносится между чанками вперёд, а backpropagation идёт только внутри чанка. Память требует места уже не под $O(Td)$ а под $O(kd)$ активаций, обновления учащаются. Цена — смещение градиента: через границы чанков он не течёт, и зависимости длиннее $k$ напрямую не обучаются, лишь косвенно через перенесённое состояние. Общая форма TBPTT($k_1$, $k_2$) — обновление каждые $k_1$ шагов с разворачиванием на $k_2$ назад

RNN можно использовать и для генерация — цикл: предсказанный $\hat y_t$ подаётся на вход шага $t+1$ (жадный argmax, сэмплирование или beam search — детали в кейсах). Состояние обновляется на месте, поэтому RNN генерирует каждый следующий токен за $O(1)$ памяти и вычислений независимо от длины уже сгенерированного. Этого свойства будут лишены трансформеры (KV-кэш растёт линейно), и его же вернут state-space модели

**Ортогональная инициализация** [(Saxe et al., 2013)](https://arxiv.org/abs/1312.6120)<br>Как было показано выше, рекуррентный процесс предполагает многократное перемножение матрицы состояния $W_{hh}$ на себя, что сильно бъет по стабильности процесса обучения. В качестве меры борьбы было предложено в качестве инициализации $W_{hh}$ вместо случайного шума брать какую-либо ортогональную матрицу

Матрица ортогональная, если все ее стоблцы - ортонормированные векторы (ортгональны друг другу и единичной нормы). У ортогональных матриц есть важное свойство - все ее сингулярные числа равны 1. А как мы выяснили выше, сингулярное число определяет стабильность процесса накполнения сигнала

Говорят, что линейная часть рекурсии ($W_{hh}$) изометрична: нормы сигнала и градиента на старте не растут и не затухают — произведения якобианов ведут себя стабильно ровно тогда, когда это важнее всего, в начале обучения

Как можно получить ортогональную матрицу: 
- QR разложение случайной Гауссовой матрицы, взять матрицу Q<br><br>*В алгебре QR разложение - представление матрицы в виде произведения ортогональной Q и диагональную R (фактически это визуализация процесса [ортогонализации Грамма-Шмидта](https://ru.wikipedia.org/wiki/%D0%9F%D1%80%D0%BE%D1%86%D0%B5%D1%81%D1%81_%D0%93%D1%80%D0%B0%D0%BC%D0%B0_%E2%80%95_%D0%A8%D0%BC%D0%B8%D0%B4%D1%82%D0%B0))<br><br>
- SVD разложение

__Weight decay__ стягивает веса к нулю, то есть уменьшает $\sigma_1(W_{hh})$ — и тем самым усугубляет затухание градиентов. Практика: исключать $W_{hh}$ из weight decay или ставить для неё коэффициент на порядок меньше, а полновесный L2 применять к входным и выходным матрицам. L1 (разреживание) на рекуррентных весах используется редко

__Dropout__<br>
Наивный dropout с новой маской на каждом шаге умножает память на цепочку случайных масок — сигнал деградирует как $(1-p)^{T}$. 

В работе [(Zaremba et al., 2014)](https://arxiv.org/abs/1409.2329) делали дропаут только на нерекуррентных связях

**Вариационный dropout** [(Gal & Ghahramani, 2016)](https://arxiv.org/abs/1512.05287) <br>идёт дальше: одна и та же маска на все шаги последовательности — на входах, выходах, рекуррентных связях и эмбеддингах; Авторы обосновывают байесовским вариационным выводом

**Zoneout** [(Krueger et al., 2016)](https://arxiv.org/abs/1606.01305)<br>вместо зануления активаций — стохастический отказ от обновления,

$$h_t=m_t\odot h_{t-1}+(1-m_t)\odot\tilde h_t,\qquad m_t\sim\operatorname{Bernoulli}(p)$$

покомпонентно. Информация не уничтожается, а замораживается; тождественная связь через замороженные компоненты дополнительно улучшает поток градиента сквозь время. На инференсе используется детерминированное среднее

**Layer Normalization** [(Ba et al., 2016)](https://arxiv.org/abs/1607.06450)<Br>нормирует преактивации по признакам в пределах одного примера и одного шага:

$$\operatorname{LN}(a)=g\odot\frac{a-\mu}{\sqrt{\sigma^{2}+\varepsilon}}+b.$$

BatchNorm в RNN неудобен: статистики зависят от батча и от позиции $t$ (при переменных длинах — от числа «живых» примеров), поведение train/test расходится. LN не зависит ни от батча, ни от $T$, ставится на преактивации ворот, стабилизирует динамику состояния — и позже станет штатным блоком трансформера.

---

## LSTM
В 1991 году Хохрайтер из Technical University Munich описал проблемы, связанные с многократным перемножением сигнала, из-за чего процесс фундаментально неустойчив с ростом длины входной последовательности. предложила принципиальную модификацию рекуррентной модели Элмана

Модель **LSTM** = Long short-term Memory [(Hochreiter & Schmidhuber, 1997)](https://doi.org/10.1162/neco.1997.9.8.1735). Помимо привычного скрытого состояния LSTM передает также отдельный сигнал, реализующий догосрочную память - "cell state", которая обновляется не умножением, а сложением: часть старого содержимого сохраняется, часть нового дописывается, и одно прибавляется к другому

Главная техническая новация - гейты (gates). Модель сама принимает решение что и когда запоминать, что нет. В качестве инструмента предложены три сигмоидных вектора из $(0,1)^d$, поэлементно умножающие информационные потоки: 0 — закрыто, 1 — открыто, между — частично. Степень пропускания вычисляется из текущего контекста $[h_{t-1};x_t]$: сеть сама решает, что помнить, что записывать и что показывать. 

Выглядит страшно, но идея простая: гейт f отвечает за выбор того, какая часть веткора памяти $c$ будет обновляться, гейт i за то, что будет обнолвться, гейт $o$ формаирует выход

<img src="img/lstm.png" width=350>

Полная система:

$$\begin{aligned}
f_t&=\sigma(W_f[h_{t-1};x_t]+b_f) && \text{гейты забывания}\\
i_t&=\sigma(W_i[h_{t-1};x_t]+b_i) && \text{гейты входа}\\
\tilde c_t&=\tanh(W_c[h_{t-1};x_t]+b_c) && \text{кандидат}\\
c_t&=f_t\odot c_{t-1}+i_t\odot\tilde c_t && \text{обновление памяти}\\
o_t&=\sigma(W_o[h_{t-1};x_t]+b_o) && \text{гейты выхода}\\
h_t&=o_t\odot\tanh(c_t) && \text{выход}
\end{aligned}$$

[ToDO]<br>
В исходной архитектуре $c_t=c_{t-1}+i_t\odot\tilde c_t$, откуда $\partial c_t/\partial c_{t-1}=I$ — карусель постоянной ошибки (constant error carousel): градиент проходит сотни шагов не затухая. Скрытое состояние $h_t$ становится рабочей проекцией памяти, а не самой памятью.

Гейтов забывания в первой версии LSTM 1997 года не было — их добавили позже [(Gers et al., 2000)](https://doi.org/10.1162/089976600300015015), когда выяснилось, что на непрерывных потоках ячейка без механизма очистки дрейфует и насыщается. $f_t$ поэлементно масштабирует старую память: конец предложения — обнулить синтаксический контекст, смена темы — стереть тематический. Градиент магистрали теперь $\partial c_t/\partial c_{t-1}=\operatorname{diag}(f_t)$: скоростью «утечки» памяти управляет сама сеть и может держать её сколь угодно близко к единице



$h_t=o_t\odot\tanh(c_t)$: в памяти много служебного — счётчики, флаги вложенности, — и не всё это релевантно текущему предсказанию; $o_t$ фильтрует, что выставить наружу и передать дальше как $h_t$

\begin{cases}
f_t &= \sigma\!\left(W_f\,[h_{t-1}, x_t] + b_f\right) \\
i_t &= \sigma\!\left(W_i\,[h_{t-1}, x_t] + b_i\right) \\
o_t &= \sigma\!\left(W_o\,[h_{t-1}, x_t] + b_o\right) \\
\tilde{c}_t &= \tanh\!\left(W_c\,[h_{t-1}, x_t] + b_c\right) \\
c_t &= f_t \odot c_{t-1} + i_t \odot \tilde{c}_t \\
h_t &= o_t \odot \tanh(c_t)
\end{cases}

__Инициализация__<br>
Из-за многократного умножения матрицы весов, корректная инциализация матриц крайне важна для RNN моделей. Отдельный вопрос, чем инициализировать смещение $b_f$. При инициализации $f_t\approx 0.5$  забывание как $0.5^{T}$ ещё до начала обучения, и градиентный сигнал о пользе долгой памяти не успевает дойти до весов.

Для борьбы с этим эффектом достаточно поставить bias побольше. нарипмер можно использовать детерминированный bias: $b_f=1\ldots 2$, тогда средняя активация будет начинаться с $\sigma(1)\approx 0.73$, то есть режим по умолчанию "помнить", забыванию — учиться

В 2015 году Sustekever с коллегами из Google провели масштабное сравнение тысяч вариантов рекуррентных архитектур через NSA [(Jozefowicz et al., 2015)](https://proceedings.mlr.press/v37/jozefowicz15.html). Что обнаружили:
- подтвердили идею еще 2000 года, в инициализации прибавлять 1 
- проранжировали гейтинг по важности: Forget, Input, Output
- LSTM / GRU арзитекутура показывает наилучшие результаты

**Peephole-соединения** [(Gers & Schmidhuber, 2000)](https://doi.org/10.1109/IJCNN.2000.861302)<br>В 2000 году те же авторы предложили модификацию своей модели - разрешиили при вычислении каждого гейта модель «подглядывает» в текущее состояние.

Модифицированые формулы выглядят так:

$$
\begin{cases}
f_t &= \sigma\!\left(W_f\,[h_{t-1}, x_t] + \textcolor{blue}{w_{cf} \odot c_{t-1}} + b_f\right) && \text{гейты забывания} \\
i_t &= \sigma\!\left(W_i\,[h_{t-1}, x_t] + \textcolor{blue}{w_{ci} \odot c_{t-1}} + b_i\right) && \text{гейты входа} \\
\tilde{c}_t &= \tanh\!\left(W_c\,[h_{t-1}, x_t] + b_c\right) && \text{состояние} \\
c_t &= f_t \odot c_{t-1} + i_t \odot \tilde{c}_t && \text{гейты забывания} \\
o_t &= \sigma\!\left(W_o\,[h_{t-1}, x_t] + \textcolor{blue}{w_{co} \odot c_t} + b_o\right) && \text{гейты выхода} \\
h_t &= o_t \odot \tanh(c_t) && \text{выход}
\end{cases}
$$

Эта модификация активизирует у модели более точное "ощущение времени": гейты видят накопленные счётчики напрямую, минуя фильтр $o_t$. Помогает в задачах точного счёта и ритма; 

Однако на большинстве NLP задач систематического выигрыша не даёт и частью стандартной реализации так и не стало

<img src="img/lstm_peephole.webp" width=350>

---

## GRU
В 2014 году исследовательская группа [(Cho et al., 2014)](https://arxiv.org/abs/1406.1078) представила модель **GRU** = gated recurrent unit . Она реализовывала тот же принцип что LSTM, но упрощала ее избыточно сложную и нагруженную архитектуру. Идея была такой: зачем нам 2 отдельных гейта для контроля записи (Input + Forget), когда это можно релазовать одним. В итоге общее кол-во гейтов сократили с трех до двух: Update + Reset

$$\begin{aligned}
z_t&=\sigma(W_z[h_{t-1};x_t]) && \text{гейты обновления} \\
r_t&=\sigma(W_r[h_{t-1};x_t]) && \text{гейты выхода} \\
\tilde h_t&=\tanh(W_h[r_t\odot h_{t-1};x_t]) && \text{выход}\\
h_t&=z_t\odot h_{t-1}+(1-z_t)\odot\tilde h_t && \text{гейты забывания}
\end{aligned}$$

Гейты обновления $z_t$ выполняют работу пары forget+input со встроенной связью $f=1-i$: новое состояние — выпуклая комбинация старого и кандидата. При $z_t\to 1$ прошлое копируется, при $z_t\to 0$ замещается новым. «Раздуться», как $c_t$ у LSTM, оно не может; выходных ворот нет: $h_t$ — одновременно и память, и выход

$r_t$ действует до вычисления кандидата: $\tilde h_t$ может «не видеть» нерелевантное прошлое и начать с чистого листа, при этом само состояние ещё не стёрто — сотрёт его или нет, решит $z_t$. Разделение труда: $r$ — краткосрочная релевантность контекста, $z$ — долгосрочный баланс старого и нового

Сравнение LSTM и GRU по кол-ву праметров

Тесты [(Chung et al., 2014)](https://arxiv.org/abs/1412.3555) показали что гейтинг важен для RNN моделей, а GRU и LSTM идут практически вровень. При этом инженерно GRU проще

Поиск по пространству вариантов [(Greff et al., 2015)](https://arxiv.org/abs/1503.04069): критичны гейты забывания и выходная нелинейность, peephole и прочие вариации погоды не делают. Эвристика: мало данных или жёсткий бюджет — GRU (меньше параметров — меньше переобучение, быстрее); большие корпуса, языковое моделирование и перевод — LSTM (чуть выше потолок качества); универсальный дефолт — LSTM с $b_f=1$.

---

## Scaling
У классичексой RNN всего 1 слой (если считаем уникальные веса и не "разворачиваем" ее). Почему бы не сделать сеть многослойной. Так появились __Stacked RNNs__. Выходная последовательность слоя $l-1$ служит входной для слоя $l$, вместо исходного текста орбабатываем промежуточные представления

$$h_t^{(l)}=\operatorname{Cell}^{(l)}\big(h_t^{(l-1)},\,h_{t-1}^{(l)}\big)$$

Cоздается иерархия признаков. Нижние слои отвечают за детекцию низкоуровенвых паттернов (орфографию и морфологию), верхние — за синтаксис и семантику. 

<img src="img/stacked_rnn.svg" width=250>

Глубокие рекуррентные стеки впервые убедительно выстрелили в распознавании речи [(Graves et al., 2013)](https://arxiv.org/abs/1303.5778); Для NLP типично использовать 2-4 слоя, на более глубоких сетях без остаточных связей (residual connections) качество не растёт

<br>

**Bi-RNN** [(Schuster & Paliwal, 1997)](https://doi.org/10.1109/78.650093)<br>Обрабатывать входную последовательность можно как слева-направо так и справа-налево. Ничто не мешает это делать одновременно, а сигналы двух проходов объединить. Так появились двунаправленные RNN. 

Bi-RNN = две независимые сети, которые читают последовательность слева направо и справа налево, а состояния комбинируются конкатенацией $h_t=[\overrightarrow{h}_t;\overleftarrow{h}_t]$, а для классификации всей последовательности — $[\overrightarrow{h}_T;\overleftarrow{h}_1]$. Мотивация: у слова есть и левый, и правый контекст (омонимия часто разрешается словами справа). Незаменима для задач Natural Understanding (энкодеры перевода), а также one-to-one задач (разметки, NER)

<img src="img/birnn.svg" width=250><br><br>

**ELMo** [(Peters et al., 2018)](https://arxiv.org/abs/1802.05365)<br>
Важное развитие подхода - модель ELMo: предобученный многослойный biLSTM-LM как источник контекстных эмбеддингов. одним из первых текстовых моделей обучаемая в парадигме Transfer Learning многоэтапного обучения (сначала универсальный pretrain, затем детальный finetune под задачу). Название продолжает традицию именования NLU моделей именами геороев улицы Сезам (BERT, ELMO, ...).

<img src="img/elmo1.png" width=500>

Обратной сети нужен конец последовательности для начала работы, поэтому применимо не во всех кейсах: онлайн-распознавание речи, синхронный перевод, автодополнение. Там либо только однонаправленные модели, либо компромиссы вроде блочной двунаправленности

<br>

**GNMT** [(Wu et al., 2016)](https://arxiv.org/abs/1609.08144)<br>
Многослойные RNN подвержены тем же проблемам, что и обычные глубокие сети. И решение то же - использование остаточных связей (__residual connection__) $h^{(l)}=h^{(l-1)}+\operatorname{Cell}^{(l)}(\cdot)$. Образцовый пример - модель GNMT, реализующая 8-слойный LSTM-стек с residual-связями

<img src="img/gnmt.png" width=250>

---

## Encoder–Decoder (Seq2Seq)

**Seq2Seq** (encoder–decoder) [(Cho et al., 2014)](https://arxiv.org/abs/1406.1078), [(Sutskever et al., 2014)](https://arxiv.org/abs/1409.3215): энкодер читает $x_{1..S}$ и сжимает её в контекстный вектор $v$ (обычно последнее состояние $h_S$); декодер — условная языковая модель $p(y_t\mid y_{<t},v)$, инициализированная $v$. Обучение — суммарная кросс-энтропия по цепному правилу. Инженерные находки Sutskever: четырёхслойные LSTM, разворот входной последовательности (первые слова источника оказываются рядом с первыми словами перевода — критические зависимости короче) и beam search при декодировании

<img src="img/seq2seq.svg" width=450>

Предложения протискивается через один вектор фиксированной размерности - он является "бутылочным горлышком". Качество перевода заметно деградирует с ростом длины фразы, исследование  [(Cho et al., 2014b)](https://arxiv.org/abs/1409.1259) это наглядно проиллюстрировало. 

Наращивание размерности вектора решает проблему локально, но такой подход совершенно не масштабируется, нужно принципиально другое архитектурное решение - например, механизм внимания (см следующий раздел)

У нас есть размеченый текст. Как его нарезать для обучения

**Teacher forcing** [(Williams & Zipser, 1989)](https://doi.org/10.1162/neco.1989.1.2.270)<br>Идея в следующем - учим модель генерировать ответ по одному токену - передаем на вход контекст из обучающей выборки и просим сгенерировать правильный токен. Иначе ошибки будут просто накапливаться

<img src="img/teacher_forcing.png" width=350>

Аналогия - автоинструктор проверяет навыки студента-водителя отдельно на каждом повороте. Если тот ошибается, инструктор и сразу же корректирует траекторию и они переходят к следюущему повороту. В конце разбирают все допущенные ошибки

**Exposure bias** [(Ranzato et al., 2015)](https://arxiv.org/abs/1511.06732)<br>
На инференсе никакой разметки (автоинструктора) нет, модель продолжать префиксы, которые могла не видеть при обучении. 

Бороться с этим эффектом можно переходя на обучение на уровне целых последовательностей (RL поверх метрики, beam-aware функции потерь), либо применяя более мягкий вариант scheduled sampling

**Scheduled sampling** [(Bengio et al., 2015)](https://arxiv.org/abs/1506.03099)<br>Чтобы побороть негативный эффект Exposure bias Bengio с коллегами предложил гибридный палн обучения - чередовать Teacher Forcing с обчением на сгенерированных токенах. Коэффицент замешивания - это параметр $\epsilon$ и он убывает по расписанию. Авторы таким образом реализуют классическуб для машинного обучения стратегию плавного перехода от Exploration к Exploitation. Расписание может быть линейным, экспоненциальным $\epsilon_i=k^{i}$ или обратно-сигмоидное $\epsilon_i=k/(k+e^{i/k})$

Разрыв смягчается, хотя целевая функция перестаёт быть корректным правдоподобием (оценка смещена)

## Механизм внимания

**Механизм внимания** [(Bahdanau et al., 2014)](https://arxiv.org/abs/1409.0473) был предложен, чтобы устранить горлышко: декодер на каждом шаге генерации "видит" все токены входной последовательности и их представления полученные энкодером

1) оцениваем "сходство" текушего состояния декодера $s_t$ и представления каждого токена $h_s$ из входной послежовательности<br>$e_{t,s}=\operatorname{score}(s_{t-1},h_s)$<br><br>
2) все посчитанные сходства нормируем softmax, получаем вектор весов, суммируемый в 1:<br>
$\alpha_{t,s}=\frac{\exp e_{t,s}}{\sum_{s'}\exp e_{t,s'}}$<br><br>
3) считаем взвешенную сумму всех представлений энкодера<br>$c_t=\sum_{s=1}^{S}\alpha_{t,s}\,h_s$

Такой подход называют мягким выравниванием: вместо жёсткого выбора слов, как в статистическом переводе (IBM-модели), — соотвествие заменяется дифференцируемым распределением $\alpha_t$

<img src="img/attention1.png" width=250>

В оригинальном варианте Бахданау score складывался аддитивно из текущего состояния $s$ и представления токена $h_j$: $$\operatorname{score}(s,h)=v_a^{\top}\tanh(W_a s+U_a h)$$

Позднее [(Luong et al., 2015)](https://arxiv.org/abs/1508.04025) заменили его на мультипликативный способ учета: $$\operatorname{score}(s,h) = s^{\top}h \quad \text{(скалярное произведение)}$$ или $\operatorname{score}(s,h) = s^{\top}W_a h$ (обощенное). Одно матричное умножение быстрее и проще; 

При больших размерностях модели $d$ скалярные произведения растут и softmax насыщается - отсюда позже добавили масштабирование на размерность задачи $q^{\top}k/\sqrt{d_k}$ в трансформере. Люонг также систематизировал global/local attention и input feeding — подачу $c_{t-1}$ на вход следующего шага

$c_t$ — взвешенное среднее по всей входной последовательности называют __контекстным вектором__. Он пересобирается заново под каждый шаг выхода; конкатенируется с состоянием декодера перед предсказанием. Контекст перестал быть константой и стал функцией запроса. В терминах, которые скоро станут каноническими: $s$ — query, а $h_s$ — keys и values.

Матрицу весов $A=[\alpha_{t,s}]\in\mathbb{R}^{T_{output}\times T_{input}}$ называют __Alignment матрицей__. Её удобно использовать для интерпретации: тепловая карта показывает, куда «смотрело» каждое выходное слово. Для близких языков — почти диагональ; перестановки вида прилагательное–существительное en–fr видны изломами. Равномерно размазанное или залипшее на одном токене внимание - типичный симптом недообученности либо ошибок маскирования

Внимание прокладывает от каждой потери $L_t$ к каждому состоянию энкодера $h_s$ путь длины $O(1)$ — в обход цепочки из десятков рекуррентных якобианов. Самый длинный и важный маршрут (выход → вход) спрямляется; затухание внутри самих цепочек RNN остаётся. 

Концептуально внимание — дифференцируемая адресация памяти по содержимому. Осталось заметить, что она справляется и без рекуррентной «несущей», — этот шаг сделает Transformer.

## Другие архитектуры

**Pointer Networks** [(Vinyals et al., 2015)](https://arxiv.org/abs/1506.03134)<br>Энкодер-декодерная модель, которая на каждом шаге генерирует не новый токен, а позицию из входной последовательности - то есть как бы «указывает» пальцем на токены входа (отсюда название). Главное, что не требуется наличие словаря. Поэтому типовое применение - комбинаторные задачи (наполнение рюкзака, построение выпуклой оболочки), а также задачи экстрактивной суммаризации (основанной на выделении релевантных блоков текста). Архитектурно используется LSTM, позиция выбирается либо детерминированно через argmax, либо через случайное сэмплирование

**CopyNet** [(Gu et al., 2016)](https://arxiv.org/abs/1603.06393)<br>
Разрешим модели иметь свой небольшой словарь и пусть на каждом шаге генерации она может выбрать токен из словаря или токен из входа. Как выбирается - складываем два скора, сортируем сумму по убыванию, применяем softmax для нормировки и сэмплируем. 

Вероятность генерации токена из словаря реализуется стандартно, через линейный слой $Wx+b$. Вероятность выбора токена $w_i$ моделируется линейным навесом над конкатенацией $[s_j, c_j, h_i]$, где s_j  c_j h_i Решает проблему OOV-имён и редких сущностей в суммаризации и диалоге

**Pointer-generator** [(See et al., 2017)](https://arxiv.org/abs/1704.04368):<br>
Та же идея, но две вероятности замешиваются с разными весами

- вероятность выбора токена $w$ из словаря считаем стандартно для seq2seq: $P_{\text{vocab}}(w) = \text{softmax}\left( \mathbf{W}_g \mathbf{s}_t + \mathbf{b}_g \right)$
- вероятность выбора токена из входной последовательности считаем, просто как его attention вес: $P_{\text{copy}}(w) = \sum_{j: x_j = w} \alpha_{t,j}$
- коэффициент замешивания двух сигналов считаем линейным слоем по всем доступным данным: $p_{\text{gen}} = \sigma\left( \mathbf{W}_p [\mathbf{s}_t; \mathbf{c}_t; \mathbf{e}(y_{t-1})] + \mathbf{b}_p \right)$

**Neural Turing Machine** [(Graves et al., 2014)](https://arxiv.org/abs/1410.5401): контроллер-LSTM плюс внешняя матрица памяти $M\in\mathbb{R}^{N\times M}$ с дифференцируемыми чтением $r_t=\sum_i w_t(i)\,M_t(i)$ и записью; адресация контентная (косинусная близость с softmax) и позиционная (сдвиги, интерполяция). Обучается алгоритмам копирования и сортировки по примерам вход–выход. 

**Differentiable Neural Computer** [(Graves et al., 2016)](https://doi.org/10.1038/nature20101) добавляет динамическую аллокацию и темпоральные связи между записями, решает графовые задачи. Концептуальный вклад — разделение вычислителя и памяти и адресация по содержимому: дальние предки современных retrieval-механизмов

**Stack-Augmented RNN** [(Joulin & Mikolov, 2015)](https://arxiv.org/abs/1503.01007): дифференцируемый стек с мягкими push/pop. RNN конечной точности — по существу конечный автомат; стек поднимает модель на ступень выше по иерархии Хомского, давая контекстно-свободные способности: $a^{n}b^{n}$, вложенные скобки, счётчики — то, что нужно синтаксису с неограниченной вложенностью.

## Cases

**Char-RNN** [(Karpathy, 2015)](http://karpathy.github.io/2015/05/21/rnn-effectiveness/):<Br>Словарь — символы (порядка 50–100), два-три слоя LSTM, softmax по алфавиту; обучение — предсказание следующего символа с truncated BPTT; генерация — авторегрессивное сэмплирование с температурой $p_i\propto\exp(z_i/\tau)$: $\tau<1$ — консервативнее, $\tau>1$ — разнообразнее. Без токенизатора модель выучивает орфографию, пунктуацию, парность скобок, структуру кода и LaTeX — наглядное свидетельство того, что предсказание следующего токена вынуждает усваивать структуру языка. Прямой идейный предок GPT

Пайплайн: токенизация → матрица эмбеддингов, инициализированная **Word2Vec** [(Mikolov et al., 2013)](https://arxiv.org/abs/1301.3781) или **GloVe** [(Pennington et al., 2014)](https://aclanthology.org/D14-1162/) (замораживать при малых данных, дообучать при больших) → biLSTM → агрегация по времени (последнее состояние либо mean/max-пулинг, строго с маской) → полносвязный слой. Типичные ошибки: пулинг по PAD-позициям и токенизация, не совпадающая со словарём эмбеддингов; против переобучения — вариационный dropout и ранняя остановка.

__Прогнозирование временных рядов__<br>
Многомерный ряд режется скользящими окнами: вход $T_{вх}$ шагов, цель — горизонт $H$. Нормализация (z-score по каналам) только по train-статистикам; валидация — строго по времени, никакого случайного сплита. Стратегии multi-step: direct ($H$ выходов сразу) и iterative (авторегрессивно; накапливает ошибку — тот же exposure bias в новом обличье). Не NLP, но канонический полигон many-to-many режима RNN

__Машинный перевод__<br>
Полный проект: сабвордная токенизация **BPE** [(Sennrich et al., 2016)](https://arxiv.org/abs/1508.07909) → biLSTM-энкодер + LSTM-декодер + внимание Луонга → teacher forcing, опционально scheduled sampling → beam search (ширина 4–10, нормализация по длине) → метрика **BLEU** [(Papineni et al., 2002)](https://aclanthology.org/P02-1040/). Чек-лист из пройденного: packing, клиппинг по норме, $b_f=1$, вариационный dropout, LayerNorm, контроль матрицы внимания.

## Debugging

Рекуррентные сети коварны тем, что их ошибки редко бывают явными. Модель может исправно запускаться, показывать убывающую кривую потерь и выдавать правдоподобные предсказания — и при этом молча работать не так, как задумано: терять контекст уже через несколько шагов, опираться на утёкшую разметку вместо реальных закономерностей или страдать от затухающих градиентов, из-за которых длинные зависимости попросту не выучиваются. В отличие от полносвязной сети, где вход и выход разделены фиксированным числом преобразований, рекуррентная модель разворачивается во времени, и её поведение складывается из сотен последовательных шагов, каждый из которых влияет на следующий. Поэтому судить о такой сети только по финальной метрике — всё равно что оценивать работу двигателя по одному лишь спидометру. Чтобы понять, что именно выучила модель, действительно ли она использует память и не маскирует ли высокий результат скрытый дефект, приходится делать её внутреннюю динамику наблюдаемой — а это и есть суть отладки.

Отладка рекуррентной сети во многом сводится к тому, чтобы сделать эту внутреннюю динамику доступной для наблюдения. Первый естественный шаг — заглянуть в скрытые состояния. Если собрать векторы $h_t$ по всему корпусу и спроецировать их в двумерную плоскость с помощью t-SNE [(van der Maaten & Hinton, 2008)](https://jmlr.org/papers/v9/vandermaaten08a.html) или UMAP [(McInnes et al., 2018)](https://arxiv.org/abs/1802.03426), а затем раскрасить точки известными метками — частью речи, тональностью, языком, — то характер получившейся картины сразу подсказывает, что именно сеть научилась кодировать. Сложившиеся кластеры выдают структуру представлений, а траектория состояний в пределах одного документа показывает, как контекст дрейфует во времени по мере чтения текста.

Также можно посмотреть, как работает гейтинг - строим распределение активаций гейтов $f_t$, $i_t$ и $z_t$ на данных датасета. Преобладающая единица говорит о долгом удержании информации, ноль — о постоянных сбросах, а выраженная бимодальность указывает на здоровую специализацию нейронов. Что за таким поведением может стоять содержательно, показал классический разбор [(Karpathy et al., 2015)](https://arxiv.org/abs/1506.02078): среди ячеек LSTM обнаружились вполне интерпретируемые — детектор нахождения внутри кавычек, счётчик длины строки, датчик глубины вложенности скобок. Иными словами, память сети действительно используется по назначению, а не остаётся мёртвым грузом.

Градиенты не всегда ведут себя ожидаемо. Полезно логировать нормы градиентов $\lVert\partial L/\partial h_t\rVert$ как функцию расстояния от точки, где вычисляются потери: экспоненциальный спад этой величины - прямое свидетельство затухания градиента вдоль последовательности. Частоту срабатывания клиппинга стоит отслеживать как самостоятельную метрику — если порог пробивается едва ли не на каждом шаге, это сигнал, что либо параметр $\theta$ выбран слишком малым, либо велика скорость обучения. А гистограммы норм градиентов, построенные послойно, дают вертикальный срез стека и помогают локализовать проблемный уровень.

Наконец, любую диагностику имеет смысл начинать с простых санити-чеков, которые отсекают грубые ошибки реализации. Модель обязана без труда переобучиться на крошечном срезе данных — одном батче или наборе коротких подпоследовательностей; если лосс при этом не идёт к нулю, дело почти наверняка в баге — неправильных масках, ошибке в packing, случайном detach, неудачной скорости обучения, — а не в «объективной сложности задачи». Полезны и обратные проверки, устроенные от противного: перестановка паддинга не должна влиять на выход, что подтверждает корректность маскирования, тогда как перемешивание меток, напротив, обязано ломать обучение — если модель всё равно учится, значит, где-то происходит утечка целевой информации.

## Критика

У рекуррентного подхода есть фундаментальное ограничение: $h_t$ невычислим раньше $h_{t-1}$. Требует $O(T)$ последовательных операций на обучении, а если конвейеры GPU/TPU простаивают, длина контекста упирается не в память, а во время. Именно это, а не качество, решило исход конкуренции архитектур.

### Сравнение с self-attention

Увидим в следующих главах, что архитектура **Transformer** [(Vaswani et al., 2017)](https://arxiv.org/abs/1706.03762) радикально меняет подход, отказываясь от прямой рекуррентности вообще. Self-attention связывает любые две позиции путём длины $O(1)$, для одной итерации достаточно одного шага.

| Механизм | Вычислений на слой | Последовательных операций | Длина пути градиента |
|---|---|---|---|
| Рекуррентный | $O(T\,d^{2})$ | $O(T)$ | $O(T)$ |
| Self-attention | $O(T^{2}\,d)$ | $O(1)$ | $O(1)$ |
| Свёрточный (ядро $k$) | $O(k\,T\,d^{2})$ | $O(1)$ | $O(\log_k T)$ |

Цена — квадратичность по длине и растущий KV-кэш на инференсе. RNN проиграли не качеством на своих масштабах, а масштабируемостью: параллелизм обучения позволил влить на порядки больше данных и параметров — началась эпоха BERT и GPT.

### Свёрточная альтернатива

**TCN** [(Bai et al., 2018)](https://arxiv.org/abs/1803.01271) развивает каузальные дилатированные свёртки **WaveNet** [(van den Oord et al., 2016)](https://arxiv.org/abs/1609.03499): рецептивное поле растёт экспоненциально с глубиной, обучение параллельно по $T$; на ряде последовательностных бенчмарков TCN обходит LSTM и GRU. Ограничение — память жёстко ограничена рецептивным полем, а состояние для стриминга — буфер длиной в это поле, не компактный вектор.

### SSM

Параллельно с равитием исследователи из Стенфорда решили переосмыслить принцип рекуррентности, взяв за сонову SSM = state space models (подробнее см главу "Альтернативные архитектуры") и описали модель **S4** [(Gu et al., 2021)](https://arxiv.org/abs/2111.00396): линейная рекурсия $h_t=\bar A h_{t-1}+\bar B x_t$, $y_t=C h_t$ (дискретизация непрерывной state-space системы). Отсутствие нелинейности между шагами даёт двойственность: обучение — свёртка с ядром $\bar K=(C\bar B,\,C\bar A\bar B,\,C\bar A^{2}\bar B,\dots)$ через FFT, параллельно; инференс — рекуррентно за $O(1)$ на шаг. Долгая память достигается не воротами, а специальной инициализацией $A$ (HiPPO); прорыв на Long Range Arena, где трансформеры проваливались. 

**Mamba** [(Gu & Dao, 2023)](https://arxiv.org/abs/2312.00752) добавляет селективность: $\bar B$, $C$ и шаг дискретизации $\Delta$ становятся функциями входа — контентно-зависимая фильтрация, функциональный наследник ворот LSTM; свёрточная форма теряется, вместо неё аппаратно-оптимизированный параллельный scan. Линейное время, константное состояние на инференсе, качество на уровне трансформеров при сопоставимых бюджетах. Параллельная линия — линейное внимание как RNN [(Katharopoulos et al., 2020)](https://arxiv.org/abs/2006.16236), **RWKV** [(Peng et al., 2023)](https://arxiv.org/abs/2305.13048) и **xLSTM** [(Beck et al., 2024)](https://arxiv.org/abs/2405.04517) с экспоненциальными воротами и матричной памятью. Круг замкнулся: рекуррентное состояние и ворота вернулись в мейнстрим как ответ на квадратичность внимания.

### Место классических RNN сегодня

Нишу определяет свойство $O(1)$-состояния: стриминг с жёсткой латентностью (онлайн-распознавание речи на **RNN-T** [(Graves, 2012)](https://arxiv.org/abs/1211.3711) годами работало в продакшене), edge-устройства и микроконтроллеры, малые датасеты и короткие последовательности (LSTM — по-прежнему сильный бейзлайн против переобучающегося трансформера), временные ряды и сенсорика. Плюс дидактика: состояние, ворота, teacher forcing, exposure bias — понятия, введённые здесь, работают во всём современном стеке.

## Инженерная оптимизация

**Gradient checkpointing** [(Chen et al., 2016)](https://arxiv.org/abs/1604.06174): хранить активации только в контрольных точках (например, каждые $\sqrt{T}$ шагов), остальные пересчитывать на обратном проходе: требует примерно +33% вычислений но зато память $O(\sqrt{T})$ вместо $O(T)$

**Mixed precision** [(Micikevicius et al., 2017)](https://arxiv.org/abs/1710.03740): вычисления в fp16/bf16, мастер-копия весов в fp32, loss scaling против underflow градиентов. Специфика RNN: тысячи последовательных шагов накапливают ошибку округления, а нормы состояний гуляют широко — узкая экспонента fp16 чревата переполнениями, bf16 надёжнее; и стоит использовать фьюзнутые cuDNN-ядра, иначе выигрыш съедается запуском множества мелких ядер.



INT8-квантизация весов для edge устройств и активаций даёт около четырёхкратной экономии памяти и заметное ускорение на CPU/NPU. Тонкость рекуррентности: ошибка квантизации состояния проходит через одну и ту же ячейку многократно и накапливается по шагам — на длинных последовательностях post-training квантизация деградирует, предпочтительнее quantization-aware training; аккумуляторы держать в int32, масштабы весов — поканальные, сигмоиды и tanh — таблицами

Экспортировать с динамическими осями: `dynamic_axes={'x': {0: 'batch', 1: 'time'}}` — иначе граф зафиксирует длины, встреченные при трассировке. LSTM/GRU отображаются во фьюзнутые операторы ONNX (кастомные ячейки — в циклы Scan/Loop, заметно медленнее). Рантаймы (ONNX Runtime, TensorRT) фьюзят операции и планируют статический граф; обязательная проверка — численный паритет с исходной моделью на батчах переменной длины: главный источник расхождений — маски и packing.

| Годы | Веха | Работы |
|---|---|---|
| 1990 | Простая рекуррентная сеть со скрытым состоянием; BPTT | Elman; Werbos |
| 1991–1994 | Диагноз затухающих и взрывающихся градиентов | Hochreiter; Bengio et al. |
| 1997 | LSTM (карусель постоянной ошибки); Bi-RNN | Hochreiter, Schmidhuber; Schuster, Paliwal |
| 2000 | Ворота забывания; peephole-соединения | Gers et al. |
| 2010–2013 | Рекуррентные языковые модели; глубокие RNN в речи; анализ градиентов и клиппинг | Mikolov et al.; Graves et al.; Pascanu et al. |
| 2013–2014 | Word2Vec, GloVe; GRU; Seq2Seq | Mikolov et al.; Cho et al.; Sutskever et al. |
| 2014–2015 | Внимание (Бахданау, Луонг); NTM; Pointer Networks; scheduled sampling | Bahdanau et al.; Graves et al.; Vinyals et al.; Bengio et al. |
| 2015–2016 | Вариационный dropout; zoneout; LayerNorm; GNMT | Gal, Ghahramani; Krueger et al.; Ba et al.; Wu et al. |
| 2017 | Transformer: отказ от рекуррентности | Vaswani et al. |
| 2018 | ELMo; TCN | Peters et al.; Bai et al. |
| 2020–2024 | Возврат рекуррентности: линейное внимание, S4, RWKV, Mamba, xLSTM | Katharopoulos et al.; Gu et al.; Peng et al.; Gu, Dao; Beck et al. |

Отличные уточняющие вопросы! Вы очень точно поймали самые тонкие места. Давайте разберем их по порядку.

---

### 1) Если бы считали loss только для последнего выхода, суммирование по k не нужно?

**Нет, это неверно.** Суммирование по \( k \) (по истории) **все равно остается**. 

В моей прошлой записи \( \frac{\partial L_t}{\partial W_{hh}} = \sum_{k=1}^{t} ... \) — это суммирование **по глубине развертки во времени** для одного-единственного шага \( t \).

*   Если у нас loss считается на **каждом** выходе (\( L = \sum_t L_t \)), то у нас будет **двойная сумма**: сначала сумма по всем моментам времени выхода \( t \), а внутри каждой — сумма по истории \( k \).
*   Если же loss считается **только для последнего выхода** \( L = L_T \), то внешняя сумма по \( t \) исчезает (остается только \( t=T \)), но **внутренняя сумма по \( k = 1..T \) остается**!

**Почему?** Потому что, чтобы вычислить, как \( W_{hh} \) повлиял на итоговый \( h_T \), нужно пройти через всю цепочку: \( h_T \to h_{T-1} \to h_{T-2} \to ... \to h_1 \). Градиент бежит по всем этим звеньям, и каждое звено \( \partial h_i / \partial h_{i-1} \) оставляет свой след. Сумма по \( k \) — это учет влияния веса на каждом из промежуточных шагов \( h_k \).

---

### 2) Почему якобиан одного шага превращается в диагональную матрицу? Поясните с размерностями.

Якобиан одного шага \( \frac{\partial h_i}{\partial h_{i-1}} \) имеет размерность \( \mathbb{R}^{n \times n} \), где \( n \) — размерность скрытого состояния.

Распишем шаг: 
\[
z_i = W_{hh} \cdot h_{i-1} + b \quad (\text{размерность } n \times 1)
\]
\[
h_i = f(z_i) \quad (\text{поэлементная функция, размерность } n \times 1)
\]

Берем производную вектора \( h_i \) по вектору \( h_{i-1} \) (правило цепочки для якобианов):
\[
\frac{\partial h_i}{\partial h_{i-1}} = 
\frac{\partial h_i}{\partial z_i} \cdot 
\frac{\partial z_i}{\partial h_{i-1}}
\]

1. **\( \frac{\partial z_i}{\partial h_{i-1}} \)** — это производная линейного преобразования \( W_{hh} \cdot h_{i-1} \) по вектору \( h_{i-1} \). Это просто **сама матрица весов** \( W_{hh} \). Размерность: \( n \times n \).

2. **\( \frac{\partial h_i}{\partial z_i} \)** — это производная поэлементной функции \( f \) по вектору \( z_i \). Так как \( h_{i}^{(j)} = f(z_i^{(j)}) \) зависит **только** от \( z_i^{(j)} \) и не зависит от других координат \( z_i^{(k \neq j)} \), все недиагональные элементы равны нулю. Это **диагональная матрица** \( D \):
\[
D = \text{diag}\big(f'(z_i^{(1)}), f'(z_i^{(2)}), \dots, f'(z_i^{(n)})\big)
\]
Размерность: \( n \times n \).

Итоговый якобиан шага (в прямом распространении):
\[
\boxed{\frac{\partial h_i}{\partial h_{i-1}} = D \cdot W_{hh}}
\]
*Размерности:* \( (n \times n) = (n \times n) \times (n \times n) \).

*(Примечание: в обратном распространении ошибки мы часто умножаем на транспонированную матрицу \( W_{hh}^T \), но суть диагональности \( D \) от этого не меняется).*

---

### 3) Почему \( L_2 \) норма называется так, если это спектральная норма?

Здесь путаница возникает из-за того, что в линейной алгебре есть **два разных объекта** с похожими названиями:

1. **\( L_2 \)-норма вектора** (или евклидова длина): \( \|x\|_2 = \sqrt{\sum x_i^2} \).
2. **Индуцированная (подчиненная) \( L_2 \)-норма матрицы** определяется через растяжение векторов:
\[
\|A\|_2 = \max_{x \neq 0} \frac{\|A x\|_2}{\|x\|_2}
\]

Она называется \( L_2 \)-нормой **матрицы**, потому что в ее определении используется \( L_2 \)-норма векторов (мы измеряем вход и выход в евклидовом расстоянии).

А **спектральной** она называется потому, что математически доказывается теорема: значение этой нормы равно **максимальному сингулярному числу** \( \sigma_{max}(A) \). А сингулярные числа — это корни из собственных чисел матрицы \( A^T A \), то есть из ее *спектра*.

**Итог:** Это **одна и та же величина**.
\[
\boxed{\|A\|_2 = \sigma_{max}(A)}
\]
Просто "спектральная норма" — это математическое название свойства (через спектр), а "\( L_2 \) норма" — это название способа ее вычисления (через евклидову длину векторов).

---

### 4) Напомните неравенство Липшица

Функция \( f(x) \) называется **\( K \)-липшицевой**, если для любых двух точек \( x \) и \( y \) из области определения выполняется:

\[
\| f(x) - f(y) \| \le K \cdot \| x - y \|
\]

Где \( K \) — константа Липшица (наименьшее такое число).

**Для линейного слоя \( f(x) = A x \):**
\[
\| A x - A y \| = \| A (x - y) \| \le \| A \|_2 \cdot \| x - y \|
\]
Следовательно, **константа Липшица линейного слоя равна его спектральной норме \( \sigma_{max}(A) \)**.

**Для нелинейности \( \tanh \):** ее производная \( \le 1 \), значит она является \( 1 \)-липшицевой.

При композиции функций (RNN) константы Липшица перемножаются. Поэтому сквозь \( T \) шагов скрытое состояние меняется не быстрее, чем \( \big(\sigma_{max}(W_{hh}) \cdot 1\big)^T \). Именно это мы и использовали для ограничения градиента.

---

### 5) Распишите размерности для производных (Якобиан) относительно \( W_{hh} \)

Это самый сложный пункт. Производная вектора по матрице — это тензор 3-го порядка (\( n \times n \times n \)), что неудобно. Поэтому на практике мы всегда считаем градиент **скалярной функции потерь \( L \)** (а не вектора \( h_t \)) по матрице весов.

Размерность \( L \) — скаляр (\( 1 \times 1 \)).  
Размерность \( W_{hh} \) — \( n \times n \).  
Значит, итоговый градиент \( \frac{\partial L}{\partial W_{hh}} \) имеет размерность **\( n \times n \)**.

Как получается эта матрица? Рассмотрим вклад шага \( k \):

1. Пусть \( \delta_k \) — это "сигнал ошибки", который пришел к скрытому состоянию \( h_k \) из будущего (через обратное распространение). Размерность \( \delta_k \): \( n \times 1 \).

2. Скрытое состояние на предыдущем шаге: \( h_{k-1} \). Размерность: \( n \times 1 \).

3. В RNN: \( h_k = f(W_{hh} \cdot h_{k-1} + b) \). Вклад в градиент по весам на этом шаге вычисляется как **внешнее произведение** вектора ошибки на вход этого слоя:
\[
\boxed{\left. \frac{\partial L}{\partial W_{hh}} \right|_{k} = \delta_k \cdot h_{k-1}^T}
\]

**Проверяем размерности:**
*   \( \delta_k \): \( [n \times 1] \)
*   \( h_{k-1}^T \): \( [1 \times n] \)
*   Их произведение: \( [n \times 1] \times [1 \times n] = [n \times n] \). Идеально совпадает с размерностью матрицы весов!

**Полный градиент** (с учетом суммирования по истории и по выходам) будет суммой таких внешних произведений для всех моментов времени \( k \):
\[
\frac{\partial L}{\partial W_{hh}} = \sum_{k} \delta_k \cdot h_{k-1}^T
\]

Именно эту формулу вы увидите в коде библиотек глубокого обучения (например, в PyTorch/TensorFlow при вызове `.backward()`). Суммирование по \( k \) в формуле BPTT как раз и отражает тот факт, что мы складываем эти матрицы \( n \times n \) для каждого временного среза.